# LFW·SurvFace 공통 Quick/Full 실험 실행기

이 노트북은 데이터셋별 legacy 노트북을 삭제하거나 서로 실행하지 않습니다. 동일한 `research/` Python 단계 함수를 한 곳에서 계획하고 순차 호출하는 상위 실행기입니다.

- `quick`: 실제 데이터와 실제 모델을 사용하되 LFW 10%, SurvFace 2%의 identity-aware·role-preserving 표본을 사용합니다.
- `full`: 전체 데이터 100%를 사용합니다.
- 모델은 `arc`, `ada`, `mag` 중 하나를 선택하고, 해당 profile과 로컬 가중치 경로를 명시합니다.
- 한 plan은 데이터셋 1개·실행 등급 1개·checkpoint 1개를 실행합니다. 세 모델 비교는 `MODEL_NAME`을 바꾸어 각각 독립 run으로 수행합니다.
- 실제 장시간 실험은 사용자가 직접 `EXECUTE=True`와 실행 확인 값을 설정해야 시작됩니다.
- 현재 공통 dispatcher는 기존 Step 4 Grad-CAM workflow를 지원합니다. 새 평가 계약의 exhaustive ADC, IVF-PQ, calibration 100/500/1,000 행렬은 아직 구현 전이므로 이 노트북의 full 완료만으로 논문 최종 비교가 되지는 않습니다.


## 1. 사용자가 조절하는 변수

`DATASET_ID`, `RUN_TIER`, `MODEL_NAME`을 선택합니다. quick 표본 비율은 `QUICK_DATA_FRACTIONS`에서 데이터셋별로 조절하며, 선택값과 기본값 차이는 실행 설정에 기록됩니다. `full`은 이 사전과 관계없이 항상 100%입니다.

`MODEL_PROFILE_BY_NAME`과 `MODEL_WEIGHT_PATHS`는 반드시 같은 학습 데이터·아키텍처의 조합이어야 합니다. `START_NEW_RUN=True`는 동일 plan의 완료 run이 이미 있는데도 독립 재실험을 의도할 때만 사용합니다.


In [1]:
from __future__ import annotations

DATASET_ID = "survface"  # "lfw" 또는 "survface"
RUN_TIER = "quick"       # "quick" 또는 "full"
QUICK_DATA_FRACTIONS = {
    "lfw": 0.3,
    "survface": 0.3,
}
SEED = 42

MODEL_NAME = "arc"  # "arc", "ada", "mag" 중 하나
MODEL_PROFILE_BY_NAME = {
    "arc": "arcface_ms1mv3_r100",
    "ada": "adaface_ms1mv3_r100",
    "mag": "magface_ms1mv2_iresnet100",
}
MODEL_WEIGHT_PATHS = {
    "arc": "models/arcface/ms1mv3_r100_backbone.pth",
    "ada": "models/adaface/adaface_ir101_ms1mv3.ckpt",
    "mag": "models/magface/magface_epoch_00025.pth",
}
# AdaFace MS1MV2 대안: profile을 adaface_ms1mv2_r100으로 바꾸고
# MODEL_WEIGHT_PATHS["ada"] = "models/adaface/adaface_ir101_ms1mv2.ckpt"
# 로 설정합니다. 현재 그 bridge profile은 Grad-CAM 실행이 비활성입니다.
MODEL_SMOKE_DEVICE = "cuda"

EXECUTE = True
ACKNOWLEDGE_LOCAL_EXECUTION = True
START_NEW_RUN = True


## 2. 프로젝트와 공통 runner 로드

현재 작업 디렉터리의 상위에서 저장소 루트를 찾습니다. 아래 셀은 실험을 시작하거나 가중치를 로드하지 않습니다.


In [2]:
from pathlib import Path
from pprint import pprint
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.pipeline_runner import (
    FULL_DATA_FRACTION,
    build_common_experiment_plan,
    inspect_common_experiment_plan,
    prepare_common_model_checkpoint,
    run_common_step4_experiment,
)
from research.runtime import ProgressReporter

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"QUICK_DATA_FRACTIONS={QUICK_DATA_FRACTIONS}")
print(f"FULL_DATA_FRACTION={FULL_DATA_FRACTION}")


PROJECT_ROOT=C:\ronbun
MODEL_NAME=arc
QUICK_DATA_FRACTIONS={'lfw': 0.3, 'survface': 0.3}
FULL_DATA_FRACTION=1.0


## 3. 선택 모델과 가중치 고정

선택한 별칭에서 profile과 가중치 경로를 가져와 checkpoint SHA-256, 전처리, target layer를 하나의 `model_uid`로 등록합니다. 등록 정보가 같으면 재사용합니다. 기존 smoke 검증 결과가 있으면 재사용하고, 없으면 최대 8장으로 forward/target-layer smoke test를 수행합니다. 전체 데이터 실험은 시작하지 않습니다.


In [3]:
if MODEL_NAME not in MODEL_PROFILE_BY_NAME or MODEL_NAME not in MODEL_WEIGHT_PATHS:
    raise ValueError("MODEL_NAME은 'arc', 'ada', 'mag' 중 하나여야 합니다.")

MODEL_PROFILE = MODEL_PROFILE_BY_NAME[MODEL_NAME]
MODEL_WEIGHT_PATH = PROJECT_ROOT / MODEL_WEIGHT_PATHS[MODEL_NAME]

MODEL_PREPARATION = prepare_common_model_checkpoint(
    project_root=PROJECT_ROOT,
    model_name=MODEL_NAME,
    model_profile=MODEL_PROFILE,
    checkpoint_path=MODEL_WEIGHT_PATH,
    run_smoke_validation=True,
    smoke_device=MODEL_SMOKE_DEVICE,
    seed=SEED,
)
pprint(MODEL_PREPARATION.as_dict(), sort_dicts=False)


{'model_name': 'arc',
 'model_profile': 'arcface_ms1mv3_r100',
 'family': 'arcface',
 'checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
 'checkpoint_sha256': 'a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b',
 'model_uid': 'arcface-7972a704552df378345f',
 'model_spec_path': 'C:\\ronbun\\runs\\step2\\model_registry\\arcface-7972a704552df378345f.json',
 'smoke_validation_status': 'reused_validated',
 'smoke_validation_path': 'C:\\ronbun\\runs\\step2\\model_validation\\arcface-7972a704552df378345f\\smoke_summary.json'}


## 4. 결정적 실행 plan 생성

manifest를 읽어 실제 선택 예정 행 수와 role/split 분포를 계산합니다. 같은 manifest hash·seed·tier라면 같은 identity 집합을 선택합니다. 이 셀도 DB나 run을 변경하지 않습니다.


In [4]:
PLAN = build_common_experiment_plan(
    project_root=PROJECT_ROOT,
    dataset_id=DATASET_ID,
    run_tier=RUN_TIER,
    seed=SEED,
    model_name=MODEL_NAME,
    model_profile=MODEL_PREPARATION.model_profile,
    model_uid=MODEL_PREPARATION.model_uid,
    model_checkpoint_path=MODEL_PREPARATION.checkpoint_path,
    quick_data_fractions=QUICK_DATA_FRACTIONS,
)
pprint(PLAN.as_dict(), sort_dicts=False)


{'plan_id': '0683688db40b7a48',
 'pipeline_id': 'common_step4_gradcam_v1',
 'dataset_id': 'survface',
 'run_tier': 'quick',
 'data_fraction': 0.3,
 'quick_data_fractions': {'lfw': 0.3, 'survface': 0.3},
 'quick_fraction_override': True,
 'seed': 42,
 'model_name': 'arc',
 'model_profile': 'arcface_ms1mv3_r100',
 'model_uid': 'arcface-7972a704552df378345f',
 'model_checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
 'evaluation_contract_id': 'face_search_evaluation_v1',
 'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
 'evaluation_contract_sha256': '4f2c4d2bf0d4d10e3d9d39d3f575e6f4154723dbf2b5bd6d0a3c8a0481c7176d',
 'base_step4_config_path': 'C:\\ronbun\\configs\\experiments\\step2_pytorch_gradcam.yaml',
 'source_rows': 463341,
 'selected_source_rows': 135529,
 'source_manifest_sha256': {'data/interim/SurvFace/training_manifest.csv': 'db5bb106a5e33ea3ebf7c9dfb06a1330c92aa2638dba8fc298318169e4bd5456',
                  

## 5. 로컬 preflight

등록 checkpoint, CUDA, ONNX Runtime provider, canonical aligned/landmark bundle, 로컬 Git source 상태를 읽기 전용으로 검사합니다. GitHub나 원격 CI는 사용하지 않습니다. quick은 dirty source를 허용하되 commit과 diff hash를 plan에 고정하고, full은 clean local commit을 요구합니다. `ready_to_execute_pipeline=False`이면 아래 실행 셀의 오류에 표시되는 실패 항목을 먼저 해결합니다.


In [5]:
PREFLIGHT = inspect_common_experiment_plan(PLAN)
pprint(PREFLIGHT, sort_dicts=False)


{'plan': {'plan_id': '0683688db40b7a48',
          'pipeline_id': 'common_step4_gradcam_v1',
          'dataset_id': 'survface',
          'run_tier': 'quick',
          'data_fraction': 0.3,
          'quick_data_fractions': {'lfw': 0.3, 'survface': 0.3},
          'quick_fraction_override': True,
          'seed': 42,
          'model_name': 'arc',
          'model_profile': 'arcface_ms1mv3_r100',
          'model_uid': 'arcface-7972a704552df378345f',
          'model_checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
          'evaluation_contract_id': 'face_search_evaluation_v1',
          'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
          'evaluation_contract_sha256': '4f2c4d2bf0d4d10e3d9d39d3f575e6f4154723dbf2b5bd6d0a3c8a0481c7176d',
          'base_step4_config_path': 'C:\\ronbun\\configs\\experiments\\step2_pytorch_gradcam.yaml',
          'source_rows': 463341,
          'selected_source_rows': 135529,

## 6. 사용자 승인 후 순차 실행 또는 재개

`EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`일 때만 실제 실험을 시작합니다. 실행 전에 위 plan과 preflight에서 데이터셋·fraction·model_uid·checkpoint 경로를 확인하십시오.

완료된 phase는 건너뛰고, 실패하거나 아직 실행하지 않은 phase부터 이어갑니다. 장시간 loop 로그는 약 10% 경계에서만 출력됩니다. 같은 plan의 완료 run이 있으면 자동으로 새 run을 만들지 않습니다.


In [6]:
if EXECUTE:
    if ACKNOWLEDGE_LOCAL_EXECUTION is not True:
        raise RuntimeError(
            "실제 실행 전 ACKNOWLEDGE_LOCAL_EXECUTION=True가 필요합니다."
        )
    if not PREFLIGHT["ready_to_execute_pipeline"]:
        CHECKS = PREFLIGHT["readiness"]["checks"]
        FAILED_CHECKS = {
            key: CHECKS.get(key)
            for key in (
                "git_policy_satisfied",
                "source_snapshot_matches",
                "cuda_available",
                "required_onnx_provider_available",
                "model_spec_verified",
            )
            if CHECKS.get(key) is not True
        }
        raise RuntimeError(f"preflight 실패: {FAILED_CHECKS}")
    PROGRESS = ProgressReporter(
        f"{DATASET_ID}/{RUN_TIER}/{MODEL_NAME}",
        heartbeat_seconds=None,
        milestone_percent=10,
    )
    EXECUTION_RESULT = run_common_step4_experiment(
        PLAN,
        execution_acknowledged=True,
        start_new_run=START_NEW_RUN,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:{RUN_TIER}:"),
    )
else:
    EXECUTION_RESULT = {
        "status": "not_started",
        "reason": "EXECUTE=False; plan과 preflight만 수행했습니다.",
    }

pprint(EXECUTION_RESULT, sort_dicts=False)


[03:16:05] survface/quick/arc | survface compressor fit | elapsed=28s | progress=50% processed=5 total=10 rate=0.18/s eta=28s family=pca
[03:16:12] survface/quick/arc | survface compressor fit | elapsed=36s | progress=60% processed=6 total=10 rate=0.17/s eta=24s family=pq profile=pq_512_m8_b8
[03:16:28] survface/quick/arc | survface compressor fit | elapsed=51s | progress=70% processed=7 total=10 rate=0.14/s eta=22s family=pq profile=pq_512_m16_b8
[03:16:54] survface/quick/arc | survface compressor fit | elapsed=1m 17s | progress=80% processed=8 total=10 rate=0.10/s eta=19s family=pq profile=pq_512_m32_b8
[03:17:42] survface/quick/arc | survface compressor fit | elapsed=2m 05s | progress=90% processed=9 total=10 rate=0.07/s eta=14s family=pq profile=pq_512_m64_b8
[03:19:17] survface/quick/arc | survface compressor fit | elapsed=3m 40s | progress=100% processed=10 total=10 rate=0.05/s eta=0s family=pq profile=pq_512_m128_b8
[03:19:31] survface/quick/arc | survface compression retrieval 

## 7. 결과 해석 경계

- `quick` 결과는 코드·DB·artifact 흐름과 경향 확인용이며 논문 최종 수치가 아닙니다.
- `full`은 전체 표본이라는 조건만 충족합니다. 동일 commit에서 LFW/SurvFace를 모두 재실행하고, 모델 UID·전처리·평가 계약 parity를 확인해야 직접 비교할 수 있습니다.
- ArcFace·AdaFace·MagFace 비교 단위는 선택한 pretrained checkpoint입니다. loss 함수 자체의 인과적 우월성 비교로 해석하지 않습니다.
- 현재 runner의 PQ 결과는 기존 reconstruction characterization입니다. exhaustive ADC와 IVF-PQ가 구현되기 전에는 PQ code-search 성능으로 주장하지 않습니다.
- Grad-CAM은 faithfulness, 교란 통제, 독립 표본 추가 예측력, 고정 FPIR 운영 개선, 비용 검증을 통과하기 전까지 보조 설명 결과입니다.
- 데이터셋별 세부 단계 디버깅은 기존 `notebooks/lfw` 또는 `notebooks/survface` 순차 노트북을 사용합니다.
